# 🔬 TrustOCT-KD: Trustworthy Lightweight Retinal OCT Classification
## Calibration-Aware Knowledge Distillation with Explainability Preservation

This notebook runs the complete research pipeline on **Google Colab**.

📌 **No Google Drive storage needed** — dataset downloads directly to Colab's free ~100GB VM disk.

---

## Step 1: GPU Check & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install required packages
!pip install -q kagglehub thop seaborn scikit-learn matplotlib tqdm opencv-python pandas

## Step 2: Clone Your GitHub Repository

In [ ]:
import os

GITHUB_REPO = "https://github.com/Gnanapravallika/TrustOCT-KD.git"

if not os.path.exists('TrustOCT-KD'):
    !git clone {GITHUB_REPO}

%cd TrustOCT-KD
!ls

## Step 3: Setup Kaggle API & Download Dataset

📌 **Dataset downloads to Colab's VM disk** (~5GB). No Google Drive needed!

### How to get your `kaggle.json`:
1. Go to [kaggle.com](https://www.kaggle.com)
2. Click your profile icon → **Settings**
3. Scroll to **API** section → Click **Create New Token**
4. It downloads `kaggle.json` to your computer
5. Upload it in the cell below

In [ ]:
# Upload your kaggle.json file
# A file upload dialog will appear — select your kaggle.json
from google.colab import files
uploaded = files.upload()

In [ ]:
# Configure Kaggle API
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API configured!")

In [ ]:
# Download Kermany OCT dataset directly to Colab VM disk (~5GB, takes 3-10 min)
!kaggle datasets download -d paultimothymooney/kermany2018 -p data/ --unzip
print("\n✅ Dataset downloaded!")

# Check disk usage
!df -h / | tail -1
!du -sh data/

In [ ]:
# Verify dataset structure & show class distribution
import os

# Find the OCT2017 directory
data_root = 'data'
for root, dirs, files_list in os.walk(data_root):
    if 'train' in dirs and 'test' in dirs:
        data_root = root
        break

print(f"Dataset root: {data_root}\n")

for split in ['train', 'test', 'val']:
    split_dir = os.path.join(data_root, split)
    if os.path.exists(split_dir):
        print(f"  {split}/")
        for cls in sorted(os.listdir(split_dir)):
            cls_path = os.path.join(split_dir, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                print(f"    {cls}: {count} images")

In [ ]:
# Update config to point to actual dataset path
import json

# Write the detected path for the pipeline to use
os.environ['TRUSTOCT_DATA_DIR'] = os.path.abspath(data_root)
print(f"Dataset path set to: {os.environ['TRUSTOCT_DATA_DIR']}")

## Step 4: Run Complete Pipeline

**One command runs everything:**
1. Train Teacher (ResNet50+MSF+CBAM)
2. Train Student baseline (MobileNetV3, no KD)
3. Calibration-Aware Knowledge Distillation
4. Full trustworthiness comparison
5. LayerCAM & AOPC explainability comparison

⏱️ **Estimated time**: ~2-4 hours on T4 GPU

In [ ]:
# ===== FULL TRAINING (2-4 hours on T4 GPU) =====
!python scripts/train_full_pipeline.py

# ===== OR: QUICK TEST (10 min, small subset, 3 epochs) =====
# !python scripts/train_full_pipeline.py --quick

## Step 5: View Results

In [ ]:
# Main comparison table (Paper Table 1)
import pandas as pd

df = pd.read_csv('outputs/results/teacher_vs_student_comparison.csv')
print("\n" + "="*60)
print(" PAPER TABLE 1: Teacher vs Student Comparison")
print("="*60)
display(df)

In [ ]:
# AOPC faithfulness comparison (Paper Table 2)
try:
    aopc_df = pd.read_csv('outputs/results/aopc_comparison.csv')
    print("\n" + "="*60)
    print(" PAPER TABLE 2: AOPC Faithfulness Comparison")
    print("="*60)
    display(aopc_df)
except:
    print("AOPC results will be available after full pipeline completes.")

## Step 6: View Publication Figures

In [ ]:
from IPython.display import Image, display
import glob

viz_files = sorted(glob.glob('outputs/visualizations/*.png'))
print(f"Generated {len(viz_files)} figures:\n")

for viz_file in viz_files:
    print(f"--- {os.path.basename(viz_file)} ---")
    display(Image(filename=viz_file, width=800))
    print()

## Step 7: Save Results (Optional)

Choose one of these options to save your results before the Colab session ends:

In [ ]:
# Option A: Download results as ZIP to your computer
!zip -r TrustOCT_Results.zip outputs/results/ outputs/visualizations/ outputs/checkpoints/

from google.colab import files
files.download('TrustOCT_Results.zip')
print("\n✅ Results ZIP downloaded to your computer!")

In [ ]:
# Option B: Save to Google Drive (only results, NOT the dataset)
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/TrustOCT_Results'
os.makedirs(save_dir, exist_ok=True)

# Only save results and figures (small files ~50MB), NOT the dataset
for folder in ['results', 'visualizations']:
    src = f'outputs/{folder}'
    dst = f'{save_dir}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Saved {folder}/")

# Save model checkpoints (~150MB)
for ckpt in glob.glob('outputs/checkpoints/*_best.pth'):
    shutil.copy(ckpt, f'{save_dir}/')
    print(f"✅ Saved {os.path.basename(ckpt)}")

print(f"\n✅ All results saved to Google Drive: {save_dir}")

---
## 📋 Paper Checklist

After running, you should have:

| Item | File | Status |
|---|---|---|
| **Table 1**: Teacher vs Student metrics | `outputs/results/teacher_vs_student_comparison.csv` | ☐ |
| **Table 2**: AOPC faithfulness | `outputs/results/aopc_comparison.csv` | ☐ |
| **Fig 2**: Confusion matrices | `outputs/visualizations/*_confusion_matrix.png` | ☐ |
| **Fig 3**: Reliability diagrams | `outputs/visualizations/*_reliability_diagram.png` | ☐ |
| **Fig 4**: LayerCAM + AOPC curves | `outputs/visualizations/Teacher_vs_Student_LayerCAM_AOPC.png` | ☐ |